In [2]:
# ==========================================
# ☀️ CUADRO DE MANDO: DEEPSOLAREYE ☀️
# ==========================================

import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from src.model import Net
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- CONFIGURACIÓN ---
# Asegúrate de que este archivo existe
MODEL_PATH = 'saved_models/model_epoch_4.pth' 
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Cargar el Modelo (Solo una vez)
print(f"Cargando cerebro de la IA en {DEVICE}...")
model = Net().to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    print("✅ Modelo cargado correctamente.")
else:
    print(f"❌ ERROR: No encuentro {MODEL_PATH}")

# 2. Función de Predicción
def analizar_panel(ruta_imagen):
    if not os.path.exists(ruta_imagen):
        return None, "Imagen no encontrada"

    # Preparar imagen
    img_original = Image.open(ruta_imagen).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    img_tensor = transform(img_original).unsqueeze(0).to(DEVICE)

    # Inferencia
    with torch.no_grad():
        output = model(img_tensor)
        suciedad = output.item()

    return img_original, suciedad

# 3. Interfaz Gráfica dentro del Notebook
def mostrar_dashboard():
    # Buscar algunas imágenes de prueba
    test_dir = 'data/raw/Solar_Panel_Soiling_Image_dataset/PanelImages'
    imagenes = [os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith('.jpg')][:5]
    
    if not imagenes:
        print("No encontré imágenes en data/raw/...")
        return

    # Crear dropdown
    dropdown = widgets.Dropdown(
        options=imagenes,
        description='Elegir Foto:',
        layout={'width': 'max-content'}
    )
    
    button = widgets.Button(description="Analizar Suciedad", button_style='primary')
    out = widgets.Output()

    def on_click(b):
        with out:
            clear_output()
            img_path = dropdown.value
            img, valor = analizar_panel(img_path)
            
            if img:
                plt.figure(figsize=(10, 6))
                plt.imshow(img)
                plt.axis('off')
                
                # Semáforo de colores
                color = 'green' if valor < 15 else 'orange' if valor < 35 else 'red'
                estado = "LIMPIO" if valor < 15 else "SUCIEDAD MEDIA" if valor < 35 else "CRÍTICO"
                
                plt.title(f"Diagnóstico IA: {valor:.2f}% de Suciedad\nEstado: {estado}", 
                          fontsize=18, color=color, fontweight='bold', backgroundcolor='white')
                plt.show()

    button.on_click(on_click)
    display(widgets.VBox([dropdown, button, out]))

# ¡Lanzar la App!
mostrar_dashboard()

Cargando cerebro de la IA en cpu...
✅ Modelo cargado correctamente.
